# Module 3: Vector Search
In this module we will explore how to use vector search in Neo4j to find similar nodes based on their properties. Neo4j has built-in support for vector search using the `vector` index type, which allows us to efficiently find nodes based on similarity with a given vector.

The database already contains a vector index named `person_embedding` on the embedding property of the `Person` nodes. The embedding property contains a vector representation of the resumé text for each person which is also stored as a string property on the node.

Lets start by importing the necessary libraries and setting up the Neo4j driver to connect to our database.

In [ ]:
from IPython.display import Markdown, display, HTML
import os
import pandas as pd
from neo4j import GraphDatabase, Result, RoutingControl
import neo4j
import markdown
from openai import OpenAI
from neo4j_viz.neo4j import from_neo4j
from dotenv import load_dotenv

load_dotenv()

uri = os.getenv("NEO4J_URI")
user = os.getenv("NEO4J_USERNAME")
password = os.getenv("NEO4J_PASSWORD")
database = os.getenv("NEO4J_DATABASE")
openai_key = os.getenv("OPENAI_KEY")


driver = GraphDatabase.driver(uri=uri, auth=(user, password))
driver.verify_connectivity()

Run the following cell to see an example of the resumé text of the `Person` nodes and the corresponding embedding (list of floats) beneath.

In [ ]:
records, summary, keys = driver.execute_query("""
MATCH (p:Person{name:'Dr. Amanda Foster'})
RETURN p.name, p.text, p.embedding
LIMIT 1
""",
database_=database,
routing=RoutingControl.READ
)

for record in records:
    display(Markdown(record.value('p.text')))
    print('Embedding: ', record.value('p.embedding'))

Next we will see how to use the vector index to find similar nodes based on their embeddings. We will use the vector index to find the most similar `Person` nodes to a given resumé text embedding. This is a common use case for vector search, where we want to find similar items based on their vector representations. Dr. Amanda Foster will be our example candidate for this search. We will use the embedding of her resumé text to find the most semantically similar candidate in the database. You can try this with any candidate in the database by using the embedding of their resumé text.

In [ ]:
df = driver.execute_query(
"""//cypher
    MATCH (p:Person{name:'Dr. Amanda Foster'})
    MATCH (person:Person)
    SEARCH person IN (
        VECTOR INDEX person_embedding
        FOR p.embedding
        LIMIT 2 // 2 since the most similar one will be the node itself
    ) SCORE AS score
    WHERE p <> person
    RETURN p.text AS source, person.text AS similar, score
""",
database_=database,
result_transformer_=neo4j.Result.to_df,
routing=RoutingControl.READ
)
df.head()

It is a bit hard to compare the resumé text in this format so lets use the `markdown` library to format the resumé text and display it in a more readable format. We will also display the similarity score for each result to see how similar the results are to our query candidate. The similarity score is a value between 0 and 1, where 1 means that the nodes are identical and 0 means that they are completely different. In practice, you will want to experiment with different similarity thresholds.

In [ ]:
for _, row in df.iterrows():
    display(HTML(f"""
    <div style="
        display: grid;
        grid-template-columns: 1fr 1fr 140px;
        gap: 16px;
        align-items: start;
        margin-bottom: 24px;
        border: 1px solid #ddd;
        padding: 12px;
    ">
        <div>{markdown.markdown(row['source'])}</div>
        <div>{markdown.markdown(row['similar'])}</div>
        <div><strong>Score</strong><br>{row['score']:.6f}</div>
    </div>
    """))

To search for similar nodes using an embedding based on text that is not already in the database, first generate an embedding for the text and use it to query the database. We can use the OpenAI API to generate an embedding for any given text. Once we have the embedding, we can use it to query the database and find similar nodes based on the vector index. This allows us to find similar candidates even if their resumé text is not exactly the same as any of the candidates in the database, but is semantically similar.

In [ ]:
client = OpenAI(api_key=openai_key)
response = client.embeddings.create(
    input="data engineer with kafka and spark skills", # change the input to test
    model="text-embedding-ada-002",
)
embedding = response.data[0].embedding

records, summary, keys = driver.execute_query(
"""//cypher
    MATCH (person:Person)
    SEARCH person IN (
        VECTOR INDEX person_embedding
        FOR $embedding
        LIMIT 3
    ) SCORE AS score
    RETURN person.text AS similar, score
""",
database_=database,
parameters_={"embedding": embedding},
routing=RoutingControl.READ
)
for idx, record in enumerate(records, start=1):
    print("result: " + str(idx))
    print('score: ', record.value('score'))
    display(Markdown(record.value('similar')))

Lets see what skills are captured in structured form for the matched candidates by adding pattern matching as a step after the vector search. Visualising the results can also help us understand the relationships between the candidates and their skills. We can use the `neo4j_viz` library to visualize the results in a graph format.

In [ ]:
client = OpenAI(api_key=openai_key)
response = client.embeddings.create(
    input="computer vision", # change the input to test
    model="text-embedding-ada-002",
)
embedding = response.data[0].embedding

result = driver.execute_query(
"""//cypher
    MATCH (person:Person)
    SEARCH person IN (
        VECTOR INDEX person_embedding
        FOR $embedding
        LIMIT 3
    ) SCORE AS score
    WITH person
    MATCH path = (person)-[:KNOWS]->(:Skill)
    RETURN path
""",
database_=database,
parameters_={"embedding": embedding},
result_transformer_=Result.graph,
routing=RoutingControl.READ
)
VG = from_neo4j(result)
VG.render()

Remember to close the Neo4j driver connection when you are done to free up resources.

In [ ]:
driver.close()

On to the next module!